In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, when, col, expr
import random

spark = SparkSession.builder.appName("DataGenerator").getOrCreate()

# Sample values
products = ["Laptop", "Mobile", "Tablet", "Watch", "Headphones"]
categories = ["Electronics", "Accessories"]
cities = ["Hyderabad", "Chennai", "Bangalore", "Mumbai", "Delhi", None]

data = []

for i in range(20000):
    order_id = random.randint(1000, 3000)  # duplicates + updates
    customer_id = f"C{random.randint(1, 500)}"
    product = random.choice(products)
    category = random.choice(categories)
    city = random.choice(cities)
    date = f"2024-01-{random.randint(1,28):02d}"
    
    # introduce NULL + negative values
    amount = random.choice([
        random.randint(1000, 60000),
        None,
        -random.randint(1000, 5000)
    ])
    
    quantity = random.randint(1, 5)

    data.append((order_id, customer_id, product, category, city, date, amount, quantity))

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df = spark.createDataFrame(data, columns)

df.show(5)



+--------+-----------+-------+-----------+---------+----------+------+--------+
|order_id|customer_id|product|   category|     city|      date|amount|quantity|
+--------+-----------+-------+-----------+---------+----------+------+--------+
|    2904|       C251| Tablet|Accessories|   Mumbai|2024-01-09| 33478|       2|
|    2861|       C475| Mobile|Electronics|Bangalore|2024-01-06| 17253|       4|
|    2550|       C253|  Watch|Electronics|Hyderabad|2024-01-26|  NULL|       1|
|    2764|       C400|  Watch|Electronics|     NULL|2024-01-21|  NULL|       4|
|    1669|       C111| Mobile|Electronics|Bangalore|2024-01-07| 34729|       1|
+--------+-----------+-------+-----------+---------+----------+------+--------+
only showing top 5 rows


In [0]:
df.write.mode("overwrite").option("header", True).csv("/Volumes/demo/bronze/bronze_data")

In [0]:
silver_df=spark.read.csv("/Volumes/demo/bronze/bronze_data", header=True,inferSchema=True)
silver_df.show()


+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|    2962|       C231|Headphones|Electronics|Bangalore|2024-01-18|  NULL|       5|
|    2216|       C397|     Watch|Accessories|  Chennai|2024-01-05|  NULL|       5|
|    1014|       C211|    Mobile|Electronics|Hyderabad|2024-01-19| 32892|       3|
|    2907|        C96|Headphones|Electronics|  Chennai|2024-01-26|  NULL|       2|
|    2099|       C326|    Tablet|Accessories|   Mumbai|2024-01-14| 29557|       2|
|    1942|       C309|    Tablet|Accessories|   Mumbai|2024-01-27| -3839|       5|
|    1142|       C202|    Laptop|Electronics|     NULL|2024-01-11| 33467|       5|
|    1550|       C484|    Laptop|Electronics|   Mumbai|2024-01-16| 32486|       2|
|    1129|       C464|    Mobile|Accessories|     NULL|2024-01-12| 12625|       1|
|   

In [0]:
silver_df.groupBy("customer_id").count().show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C309|   39|
|       C159|   37|
|       C407|   47|
|       C470|   32|
|       C311|   35|
|        C81|   46|
|        C31|   38|
|       C362|   49|
|       C262|   33|
|       C388|   55|
|       C331|   40|
|       C130|   40|
|       C198|   40|
|       C415|   45|
|        C97|   46|
|       C385|   37|
|        C38|   41|
|       C158|   32|
|       C491|   52|
|       C187|   35|
+-----------+-----+
only showing top 20 rows


In [0]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- amount: integer (nullable = true)
 |-- quantity: integer (nullable = true)



In [0]:
print(silver_df.count())

20000


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, to_date


# Window: customer + product
window_spec = Window.partitionBy("customer_id", "product") \
                    .orderBy(col("date").desc())

# Rank
df_with_rank = silver_df.withColumn("rn", row_number().over(window_spec))

# Keep latest per product
final_df = df_with_rank.filter(col("rn") == 1).drop("rn")

final_df.show()
print(final_df.count())

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|    2077|         C1|Headphones|Electronics|    Delhi|2024-01-27| -1153|       5|
|    1211|         C1|    Laptop|Electronics|  Chennai|2024-01-28|  NULL|       2|
|    2477|         C1|    Mobile|Electronics|   Mumbai|2024-01-28|  NULL|       5|
|    1405|         C1|    Tablet|Accessories|    Delhi|2024-01-14| 33629|       5|
|    1826|         C1|     Watch|Electronics|Hyderabad|2024-01-26| -2760|       2|
|    2518|        C10|Headphones|Accessories|   Mumbai|2024-01-26| 20531|       5|
|    1717|        C10|    Laptop|Electronics|    Delhi|2024-01-25| 19428|       2|
|    2896|        C10|    Mobile|Electronics|   Mumbai|2024-01-27|  NULL|       3|
|    2906|        C10|    Tablet|Accessories|Hyderabad|2024-01-28| 51563|       2|
|   

In [0]:
final_df.filter(col('amount').isNull()).count()

890

In [0]:
final_df.filter(col('quantity').isNull()).count()

0

In [0]:
final_df=final_df.fillna({'city':'unknown',
                          'amount':0
                          })

In [0]:
final_df=final_df.filter(col("amount") >=0)
print(final_df.count())


1688


In [0]:
final_df.write.mode("append").option("header", True).csv("/Volumes/demo/silver/silver_data")

In [0]:
gold_df=spark.read.csv("/Volumes/demo/silver/silver_data", header=True,inferSchema=True)
gold_df.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|    1211|         C1|    Laptop|Electronics|  Chennai|2024-01-28|     0|       2|
|    2477|         C1|    Mobile|Electronics|   Mumbai|2024-01-28|     0|       5|
|    1405|         C1|    Tablet|Accessories|    Delhi|2024-01-14| 33629|       5|
|    2518|        C10|Headphones|Accessories|   Mumbai|2024-01-26| 20531|       5|
|    1717|        C10|    Laptop|Electronics|    Delhi|2024-01-25| 19428|       2|
|    2896|        C10|    Mobile|Electronics|   Mumbai|2024-01-27|     0|       3|
|    2906|        C10|    Tablet|Accessories|Hyderabad|2024-01-28| 51563|       2|
|    1594|        C10|     Watch|Accessories|    Delhi|2024-01-25| 46842|       1|
|    2617|       C100|    Laptop|Accessories|Hyderabad|2024-01-26|     0|       1|
|   

In [0]:
gold_df.write.mode("overwrite").option("header", True).csv("/Volumes/demo/gold/gold_data")